# 18 — RS126 Soft Priority ve Skor Deneyleri

Önceki deneyde `RS126 > 0` sert filtresi Validation döneminde umut verici olsa da zaman bloklarında stabil bulunmadı. Bu notebook RS126'yı daha yumuşak biçimde kullanır.

İki mekanizma denenir:

1. **Aday önceliği:** Mevcut `AL` kümesi değişmez; göreceli gücü yüksek hisseler portföy slotlarında öncelik kazanır.
2. **Marjinal skor ayarı:** RS126 yalnızca Robot skoruna `+1` veya `-1` ekleyerek sınırdaki sinyalleri etkiler.

> Üretimdeki Baseline Robot ve günlük sinyal sistemi değiştirilmez.

## Neden yalnızca `RS126` ile sıralamıyoruz?

Aynı tarihte bütün hisselerden aynı BIST100 getirisi çıkarıldığı için:

```text
RS126 = Hisse RET126 - aynı günün BIST100 RET126 değeri
```

Ham `RS126` sıralaması, o gün için ham `RET126` sıralamasıyla aynıdır. Baseline zaten `RET126` kullanıyor. Bu nedenle deney, RS126'nın **Score ile ağırlıklı birleşimini** ve **yumuşak skor ayarlarını** test eder.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import build_market_regime
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.rs126_soft_experiments import (
    add_baseline_deltas,
    baseline_variant,
    build_validation_acceptance_table,
    build_validation_variants,
    default_soft_variants,
    evaluate_soft_variant,
    prepare_soft_rs126_dataset,
    run_soft_grid,
    save_soft_rs126_artifacts,
    select_development_family_winners,
    summarize_block_stability,
    variants_by_name,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

## 1. Tam geçmiş araştırma verisini hazırla

In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(
    market_features
)

prepared_prices = prepare_soft_rs126_dataset(
    stock_features=stock_features,
    market_features=market_features,
    market_regime=market_regime,
    strategy_config=FINAL_STRATEGY_CONFIG,
)

print("Hazırlanan veri:", prepared_prices.shape)
print(
    "Tarih aralığı:",
    prepared_prices["Date"].min(),
    "→",
    prepared_prices["Date"].max(),
)

display(
    prepared_prices[
        [
            "Date",
            "Ticker",
            "Baseline_Score",
            "Baseline_Signal",
            "RET_126",
            "MARKET_RET_126",
            "RS_126",
            "RS126_Percentile",
            "RS126_Centered_Percentile",
        ]
    ].tail()
)

## 2. Ham RET126 ve RS126 sıralama eşdeğerliği kontrolü

Aynı gün içindeki Spearman sıralama korelasyonunun teorik olarak `1,0` olması beklenir.

In [ ]:
rank_correlations = (
    prepared_prices.dropna(
        subset=["RET_126", "RS_126"]
    )
    .groupby("Date")
    .apply(
        lambda frame: frame[
            ["RET_126", "RS_126"]
        ].corr(
            method="spearman"
        ).iloc[0, 1],
        include_groups=False,
    )
)

display(rank_correlations.describe())

print(
    "1,0'a eşit gün oranı:",
    rank_correlations.eq(1.0).mean(),
)

## 3. Development — yumuşak mekanizma seçimi

İki aile vardır:

```text
RS126_Ranking
→ AL kümesini değiştirmez
→ Score + ağırlık × merkezlenmiş RS126 yüzdelik sırası

RS126_Score_Adjustment
→ yalnızca sınırdaki Robot skorlarını +1 / -1 etkiler
```

Development sonucu görülmeden yeni ağırlık veya eşik eklenmemelidir.

In [ ]:
DEVELOPMENT_PERIOD = {
    "Development": (
        "2018-01-01",
        "2022-12-31",
    )
}

development_variants = (
    default_soft_variants()
)

development_results = run_soft_grid(
    prepared_prices=prepared_prices,
    variants=development_variants,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    periods=DEVELOPMENT_PERIOD,
)

development_view = [
    "Variant",
    "Family",
    "CAGR_%",
    "Max_Drawdown_%",
    "Profit_Factor",
    "Sharpe",
    "Calmar",
    "Trade_Count",
    "Signal_Change_Count",
    "Promoted_To_AL_Count",
    "Demoted_From_AL_Count",
    "Crowded_Signal_Days",
]

display(
    development_results[
        development_view
    ].sort_values(
        ["Calmar", "CAGR_%"],
        ascending=False,
    )
)

## 4. Development aile kazananları

Her aileden en fazla bir varyant seçilir. Seçim yalnızca Development dönemine dayanır.

In [ ]:
development_winners = (
    select_development_family_winners(
        development_results
    )
)

winner_view = [
    "Variant",
    "Family",
    "CAGR_%",
    "CAGR_Delta_pp",
    "Max_Drawdown_%",
    "Max_DD_Improvement_pp",
    "Profit_Factor",
    "Calmar",
    "Trade_Count",
    "Trade_Fraction",
    "Signal_Change_Count",
]

display(
    development_winners[
        winner_view
    ]
)

validation_variants = (
    build_validation_variants(
        development_winners=development_winners,
        development_variants=development_variants,
    )
)

print(
    "Validation aday sayısı:",
    len(validation_variants),
)
print(
    *[variant.name for variant in validation_variants],
    sep="\n- ",
)

## 5. Validation — kabul veya ret

Validation adayları:

- Baseline
- Development'ta seçilen ranking varyantı
- Development'ta seçilen score-adjustment varyantı
- İki aile kazananının kombinasyonu

Kombinasyon Validation sonucuna bakılmadan oluşturulur.

In [ ]:
VALIDATION_PERIOD = {
    "Validation": (
        "2023-01-01",
        "2024-12-31",
    )
}

validation_results = run_soft_grid(
    prepared_prices=prepared_prices,
    variants=validation_variants,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    periods=VALIDATION_PERIOD,
)

acceptance_table = (
    build_validation_acceptance_table(
        validation_results
    )
)

acceptance_view = [
    "Variant",
    "Family",
    "CAGR_%",
    "CAGR_Delta_pp",
    "Max_Drawdown_%",
    "Max_DD_Improvement_pp",
    "Profit_Factor",
    "Profit_Factor_Delta",
    "Sharpe",
    "Calmar",
    "Calmar_Improvement_%",
    "Trade_Count",
    "Trade_Fraction",
    "Signal_Change_Count",
    "Promoted_To_AL_Count",
    "Demoted_From_AL_Count",
    "Return_Case_Accepted",
    "Risk_Case_Accepted",
    "Enhancement_Accepted",
]

display(
    acceptance_table[
        acceptance_view
    ]
)

### Kabul kuralları

**Getiri odaklı**

```text
CAGR farkı             >= +1,5 yüzde puan
Drawdown               kötüleşmemeli
Calmar iyileşmesi      >= %5
Profit Factor          düşmemeli
İşlem sayısı           Baseline'ın en az %70'i
```

**Risk odaklı**

```text
CAGR farkı             >= -1,0 yüzde puan
Drawdown iyileşmesi    >= 2 yüzde puan
Calmar iyileşmesi      >= %8
Profit Factor          Baseline'ın en az %95'i
İşlem sayısı           Baseline'ın en az %70'i
```

In [ ]:
accepted_names = acceptance_table.loc[
    acceptance_table[
        "Enhancement_Accepted"
    ],
    "Variant",
].tolist()

validation_map = variants_by_name(
    validation_variants
)

accepted_variants = [
    validation_map[name]
    for name in accepted_names
]

print(
    "Validation kabul edilen varyantlar:",
    accepted_names or "Yok",
)

## 6. Zaman bloğu dayanıklılığı

Validation kabul edilen adaylar dört ayrı zaman bloğunda test edilir. En az üç blokta olumlu katkı beklenir.

In [ ]:
STABILITY_PERIODS = {
    "2019-2020": (
        "2019-01-01",
        "2020-12-31",
    ),
    "2021-2022": (
        "2021-01-01",
        "2022-12-31",
    ),
    "2023": (
        "2023-01-01",
        "2023-12-31",
    ),
    "2024": (
        "2024-01-01",
        "2024-12-31",
    ),
}

block_results = None
block_summary = None

if accepted_variants:
    block_results = run_soft_grid(
        prepared_prices=prepared_prices,
        variants=(
            [baseline_variant()]
            + accepted_variants
        ),
        strategy_config=FINAL_STRATEGY_CONFIG,
        portfolio_config=FINAL_PORTFOLIO_CONFIG,
        periods=STABILITY_PERIODS,
    )

    block_summary = summarize_block_stability(
        block_results
    )

    display(block_summary)
else:
    print(
        "Validation kabul edilen aday olmadığı için "
        "blok analizi çalıştırılmadı."
    )

## 7. Validation görsel karşılaştırması

In [ ]:
plot_data = acceptance_table.copy()

plt.figure(figsize=(12, 7))
plt.scatter(
    plot_data["Max_Drawdown_%"],
    plot_data["CAGR_%"],
    s=(
        plot_data["Trade_Fraction"]
        .fillna(1.0)
        .clip(lower=0)
        * 180
    ),
)

for row in plot_data.itertuples():
    plt.annotate(
        row.Variant,
        (
            row._asdict()["Max_Drawdown_%"],
            row._asdict()["CAGR_%"],
        ),
        fontsize=8,
        xytext=(4, 4),
        textcoords="offset points",
    )

baseline_row = acceptance_table.loc[
    acceptance_table["Variant"].eq(
        "Baseline"
    )
].iloc[0]

plt.axhline(
    baseline_row["CAGR_%"],
    linestyle="--",
    label="Baseline CAGR",
)
plt.axvline(
    baseline_row["Max_Drawdown_%"],
    linestyle="--",
    label="Baseline Drawdown",
)

plt.title(
    "Validation — Soft RS126 Adayları"
)
plt.xlabel("Maksimum Drawdown (%)")
plt.ylabel("CAGR (%)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. En iyi kabul edilen adayın Validation equity karşılaştırması

In [ ]:
best_candidate_name = None

accepted_table = acceptance_table.loc[
    acceptance_table[
        "Enhancement_Accepted"
    ]
].copy()

if not accepted_table.empty:
    best_candidate_name = accepted_table.iloc[
        0
    ]["Variant"]
    best_candidate = validation_map[
        best_candidate_name
    ]

    _, baseline_equity, _ = (
        evaluate_soft_variant(
            prepared_prices=prepared_prices,
            variant=baseline_variant(),
            strategy_config=(
                FINAL_STRATEGY_CONFIG
            ),
            portfolio_config=(
                FINAL_PORTFOLIO_CONFIG
            ),
            start="2023-01-01",
            end="2024-12-31",
        )
    )

    _, candidate_equity, _ = (
        evaluate_soft_variant(
            prepared_prices=prepared_prices,
            variant=best_candidate,
            strategy_config=(
                FINAL_STRATEGY_CONFIG
            ),
            portfolio_config=(
                FINAL_PORTFOLIO_CONFIG
            ),
            start="2023-01-01",
            end="2024-12-31",
        )
    )

    plt.figure(figsize=(13, 7))
    plt.plot(
        baseline_equity["Date"],
        baseline_equity["Equity"],
        label="Baseline",
    )
    plt.plot(
        candidate_equity["Date"],
        candidate_equity["Equity"],
        label=best_candidate_name,
    )
    plt.title(
        "Validation Equity — Baseline ve Soft RS126 Adayı"
    )
    plt.xlabel("Tarih")
    plt.ylabel("Portföy Değeri (TL)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print(
        "Equity karşılaştırması için kabul edilen aday yok."
    )

## 9. Audit — varsayılan olarak kapalı

Audit yalnızca Validation ve blok dayanıklılığı tamamlandıktan, aday adı kilitlendikten sonra açılmalıdır.

> 2025+ dönemi daha önce incelendiği için kusursuz bir holdout değildir.

In [ ]:
RUN_AUDIT = True

audit_results = None

if RUN_AUDIT:
    if block_summary is not None:
        stable_names = block_summary.loc[
            block_summary[
                "Stable_Across_Blocks"
            ],
            "Variant",
        ].tolist()
    else:
        stable_names = []

    audit_variants = [
        validation_map[name]
        for name in stable_names
    ]

    if audit_variants:
        audit_end = min(
            prepared_prices["Date"].max(),
            market_prices["Date"].max(),
        ).strftime("%Y-%m-%d")

        audit_results = run_soft_grid(
            prepared_prices=prepared_prices,
            variants=(
                [baseline_variant()]
                + audit_variants
            ),
            strategy_config=FINAL_STRATEGY_CONFIG,
            portfolio_config=FINAL_PORTFOLIO_CONFIG,
            periods={
                "Audit_2025_Plus": (
                    "2025-01-01",
                    audit_end,
                )
            },
        )

        display(audit_results)
    else:
        print(
            "Audit için stabil kabul edilen aday yok."
        )
else:
    print(
        "Audit kapalı. Karar kilitlenmeden açmayın."
    )

## 10. Sonuçları kaydet

Çıktılar `.gitignore` kapsamındaki yerel sonuç klasörüne yazılır.

In [ ]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "rs126_soft_experiments"
)

metadata = {
    "production_baseline_changed": False,
    "experiment": (
        "RS126 soft ranking and marginal score adjustment"
    ),
    "development_period": DEVELOPMENT_PERIOD,
    "validation_period": VALIDATION_PERIOD,
    "stability_periods": STABILITY_PERIODS,
    "audit_enabled": RUN_AUDIT,
    "strategy_config": asdict(
        FINAL_STRATEGY_CONFIG
    ),
    "portfolio_config": asdict(
        FINAL_PORTFOLIO_CONFIG
    ),
    "data_start": (
        prepared_prices["Date"]
        .min()
        .isoformat()
    ),
    "data_end": (
        prepared_prices["Date"]
        .max()
        .isoformat()
    ),
    "important_note": (
        "Raw same-day RS126 ordering is identical "
        "to RET126 ordering. The experiment therefore "
        "uses score blending and marginal score changes."
    ),
}

artifact_paths = save_soft_rs126_artifacts(
    output_directory=OUTPUT_DIR,
    development_results=development_results,
    development_winners=development_winners,
    validation_results=validation_results,
    acceptance_table=acceptance_table,
    block_results=block_results,
    block_summary=block_summary,
    audit_results=audit_results,
    metadata=metadata,
)

for name, path in artifact_paths.items():
    print(name, "→", path)

## Karar çerçevesi

- **Validation kabulü yoksa:** Baseline korunur.
- **Validation kabulü var, blok stabilitesi yoksa:** Baseline korunur.
- **Ranking varyantı başarılıysa:** Günlük `AL` kümesi aynı kalır; yalnızca slot önceliği değişir.
- **Score-adjustment varyantı başarılıysa:** Yalnızca sınırdaki skorlar etkilenir.
- **Kombinasyon başarılıysa:** Her iki mekanizma birlikte ayrıca Audit'e taşınabilir.

Hiçbir sonuç günlük sinyal kodunu otomatik değiştirmez.